## Import Libraries

In [73]:
import os

os.environ["HSA_OVERRIDE_GFX_VERSION"] = "10.3.0"

In [74]:
import numpy as np

import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

## Prepare train dataset

In [75]:
# Transform to turn into tensor and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307),(0.3081,)) # mean y std
])

In [76]:
# Load the dataset and normalize it
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

## Prepare test dataset

In [77]:
transform_test = transforms.ToTensor()

In [78]:
# Load the test dataset
test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=False,
    transform=transform_test
)

## Data Loaders

In [79]:
# DataLoaders
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=1028,
    num_workers=4,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=2056,
    num_workers=4,
    shuffle=True,
    drop_last=False
)

## Feed Forward Network 

In [80]:
import torch

print(f"¿GPU disponible?: {torch.cuda.is_available()}")
print(f"Nombre de la GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

¿GPU disponible?: True
Nombre de la GPU: AMD Radeon RX 6600


In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

Entrenando en: cuda


In [94]:
def createTheMNISTnet(nUnits, nLayers):

    class mnistNet(nn.Module):
        def __init__(self, nUnits, nLayers):
            super().__init__()

            # Dictionary to store layers
            self.layers = nn.ModuleDict()
            self.nLayers = nLayers
            
            # input layer
            self.layers['input'] = nn.Linear(784, nUnits) # 784 pixels

            # hidden layer
            for i in range(nLayers):
                self.layers[f'hidden{i}'] = nn.Linear(nUnits, nUnits)

            # output layer
            self.layers['output'] = nn.Linear(nUnits,10)

        def forward(self,x):
            x = torch.flatten(x, start_dim=1)
            
            x = self.layers['input'](x)

            for i in range(self.nLayers):
                x = F.relu( self.layers[f'hidden{i}'](x) )

            x = self.layers['output'](x)
            return x

    net = mnistNet(nUnits, nLayers)

    lossfun = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(net.parameters(), lr=0.001) # try with Adam later

    return net, lossfun, optimizer

In [95]:
nUnitsPerLayer = 12 
nLayers = 4
net = createTheMNISTnet(nUnitsPerLayer, nLayers)
net

(mnistNet(
   (layers): ModuleDict(
     (input): Linear(in_features=784, out_features=12, bias=True)
     (hidden0): Linear(in_features=12, out_features=12, bias=True)
     (hidden1): Linear(in_features=12, out_features=12, bias=True)
     (hidden2): Linear(in_features=12, out_features=12, bias=True)
     (hidden3): Linear(in_features=12, out_features=12, bias=True)
     (output): Linear(in_features=12, out_features=10, bias=True)
   )
 ),
 CrossEntropyLoss(),
 SGD (
 Parameter Group 0
     dampening: 0
     differentiable: False
     foreach: None
     fused: None
     lr: 0.001
     maximize: False
     momentum: 0
     nesterov: False
     weight_decay: 0
 ))

## Training

In [96]:
def function2trainTheModel(nUnits, nLayers):

    numepochs = 60

    net, lossfun, optimizer = createTheMNISTnet(nUnits, nLayers)
    net.to(device)

    losses = torch.zeros(numepochs)
    trainAcc = [] 
    testAcc = []

    for epochi in range(numepochs):

        # ===== Train =====
        batchAcc = []
        batchLoss = []
        for X,y in train_loader:

            X, y = X.to(device), y.to(device)
            
            # forward
            yHat = net(X)
            loss = lossfun(yHat, y)

            # back
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # loss
            batchLoss.append( loss.item() )

            # accuracy
            matches = torch.argmax(yHat, axis=1) == y # predictions
            matchesNumeric = matches.float() # to numeric
            accuracyPct = 100*torch.mean(matchesNumeric) # pct of mean
            batchAcc.append( accuracyPct.item() ) # add

        trainAcc.append( np.mean(batchAcc) )

        losses[epochi] = np.mean(batchLoss)

        # ===== Test =====
        X,y = next(iter(test_loader))
        X, y = X.to(device), y.to(device)
        with torch.no_grad():
            yHat = net(X)

        acc = 100*torch.mean( (torch.argmax(yHat, axis=1)==y).float() )
        testAcc.append( acc.item() )
        
        print(f'Epoch: {epochi} \f trainAcc: {trainAcc[epochi]} \f testAcc: {testAcc[epochi]}')

    return trainAcc, testAcc, losses, net

In [ ]:
numlayers = range(1,4)
numunits = np.arange(50, 251, 50)

accuracies = np.zeros( (2,len(numunits),len(numlayers)) )

for unitidx in range(len(numunits)):
    for layeridx in range(len(numlayers)):

        trainAcc, testAcc, losses, net = function2trainTheModel(numunits[unitidx], numlayers[layeridx])

        accuracies[0,unitidx,layeridx] = np.mean(trainAcc[-5:])
        accuracies[1,unitidx,layeridx] = np.mean(testAcc[-5:])

        print(f'Finished units {unitidx+1}/{len(numunits)} and layers {layeridx+1}/{len(numlayers)}')

/home/alexpy/Documents/DLNotes/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:310.)
  return F.linear(input, self.weight, self.bias)


Epoch: 0  trainAcc: 15.829195778945397  testAcc: 16.050582885742188
Epoch: 1  trainAcc: 19.08795085446588  testAcc: 17.461090087890625
Epoch: 2  trainAcc: 22.645243282975823  testAcc: 20.573930740356445
Epoch: 3  trainAcc: 26.427277170378588  testAcc: 22.762645721435547
Epoch: 4  trainAcc: 30.777874354658454  testAcc: 23.686771392822266
Epoch: 5  trainAcc: 35.26935429408633  testAcc: 25.53502082824707
Epoch: 6  trainAcc: 39.45223387356462  testAcc: 25.972763061523438
Epoch: 7  trainAcc: 43.24600870855923  testAcc: 30.88521385192871
Epoch: 8  trainAcc: 46.36555783501987  testAcc: 32.538909912109375
Epoch: 9  trainAcc: 48.94673308010759  testAcc: 34.82490158081055
Epoch: 10  trainAcc: 51.08177928266854  testAcc: 36.33268356323242
Epoch: 11  trainAcc: 52.91493297445363  testAcc: 41.92607116699219
Epoch: 12  trainAcc: 54.40762033133671  testAcc: 44.45525360107422
Epoch: 13  trainAcc: 55.534683293309705  testAcc: 48.54085922241211
Epoch: 14  trainAcc: 56.5929827

Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 227, in __call__
    res = self._callback(*self._args, **self._kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/util.py", line 136, in _remove_temp_dir
    rmtree(tempdir, onerror=onerror)
  File "/usr/lib/python3.12/shutil.py", line 796, in rmtree
    onexc(os.rmdir, path, err)
  File "/usr/lib/python3.12/shutil.py", line 764, in onexc
    return onerror(func, path, exc_info)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/shutil.py", line 794, in rmtree
    os.rmdir(path, dir_fd=dir_fd)
OSError: [Errno 39] Directory not empty: '/tmp/pymp-bqy7tyre'


Epoch: 20  trainAcc: 52.85623155791184  testAcc: 48.054473876953125
Epoch: 21  trainAcc: 53.050784209678916  testAcc: 52.188716888427734
Epoch: 22  trainAcc: 53.26378532935833  testAcc: 52.77237319946289
Epoch: 23  trainAcc: 53.45666109282395  testAcc: 53.161476135253906
Epoch: 24  trainAcc: 53.73004058311726  testAcc: 55.009727478027344
Epoch: 25  trainAcc: 54.025224356815734  testAcc: 54.18288040161133
Epoch: 26  trainAcc: 54.36065976373081  testAcc: 55.30155944824219
Epoch: 27  trainAcc: 54.784985114788185  testAcc: 57.29572296142578
Epoch: 28  trainAcc: 55.47095061992777  testAcc: 58.803504943847656
Epoch: 29  trainAcc: 56.09989212299215  testAcc: 57.83073806762695
Epoch: 30  trainAcc: 56.89655165836729  testAcc: 59.58171463012695
Epoch: 31  trainAcc: 57.82570792888773  testAcc: 61.86770248413086
Epoch: 32  trainAcc: 58.941030502319336  testAcc: 62.354087829589844
Epoch: 33  trainAcc: 59.89198993814403  testAcc: 62.15953063964844
Epoch: 34  trainAcc: 61

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(15,6))

az[0].plot(numunits, accuracies[0,:,:], 'o-', markerfacecolor='w', markersize=9)
ax[1].plot(numunits, accuracies[1,:,:],'o-', markerfacecolor='w', markersize=9)

for i in range(2):
    ax[i].legend(numlayers)
    ax[i].set_ylabel('Accuracy')
    ax[i].set_xlabel('Number of hidden units')
    ax[i].set_title(['Train' if i==0 else 'Test' ][0])
plt.show()